# Algorithm Showdown: 6 ML Algorithms on Cancer Data

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/genieincodebottle/algorithm-showdown-6-ml-algorithms-on-cancer-data)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/genieincodebottle/aiml-companion/blob/main/projects/algorithm-showdown/notebooks/Algorithm_Showdown.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-View_Source-blue?logo=github)](https://github.com/genieincodebottle/aiml-companion/tree/main/projects/algorithm-showdown)

**Author:** [genieincodebottle](https://github.com/genieincodebottle) | **Dataset:** [UCI Breast Cancer Wisconsin](https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic)) | **Last Updated:** March 2026

---

## What You Will Learn

In this project, you will:
1. Train **6 competing ML algorithms** on real clinical data
2. Compare them using **5-fold cross-validation** across 5 metrics
3. **Tune the decision threshold** to achieve high recall for cancer screening
4. Use **SHAP** to explain why the model makes each prediction

<div class="alert alert-info">
<strong>Dataset:</strong> UCI Breast Cancer Wisconsin - 569 patients, 30 cell nucleus features, binary classification (malignant vs benign)<br>
<strong>Time:</strong> ~30-45 minutes | <strong>Difficulty:</strong> Intermediate
</div>

---

<a id="toc"></a>
## Table of Contents

1. [Setup](#setup) - Install dependencies
2. [Load and Explore Data](#load-data) - Understand the dataset
3. [Build 6 Competing Algorithms](#build-algorithms) - Define scikit-learn pipelines
4. [Cross-Validation Comparison](#cross-val) - Compare all algorithms on 5 metrics
5. [Threshold Tuning for Cancer Screening](#threshold-tuning) - Optimize for recall
6. [SHAP Explainability](#shap) - Understand why the model predicts what it predicts
7. [SHAP Visualizations](#shap-viz) - Bar chart and beeswarm plots
8. [Key Takeaways](#conclusion) - Summary and next steps

---

<a id="setup"></a>
## 1. Setup

Run the cell below to install all required packages. This takes about 30 seconds on Colab.

In [ ]:
!pip install scikit-learn "xgboost>=2.0,<3.0" "shap>=0.45,<1.0" matplotlib pandas numpy -q

# Verify versions (exact metrics depend on library versions)
import sklearn, xgboost
print(f'scikit-learn: {sklearn.__version__}')
print(f'xgboost: {xgboost.__version__}')

---

<a id="load-data"></a>
## 2. Load and Explore Data

We are using the **UCI Breast Cancer Wisconsin (Diagnostic)** dataset, a well-known benchmark in medical ML. Each sample has 30 numeric features computed from digitized images of cell nuclei:

- **10 base measurements**: radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension
- **3 statistics each**: mean, standard error, and worst (largest) value

<div class="alert alert-info">
<strong>Target variable:</strong> <code>0 = malignant</code> (cancer), <code>1 = benign</code> (not cancer)
</div>

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Classes: {np.bincount(y)} (0=malignant, 1=benign)')
print(f'Class balance: {y.mean():.1%} benign')

<div class="alert alert-success">
<strong>What you should see above:</strong>
<ul>
<li>569 samples with 30 features each</li>
<li>~63% benign, ~37% malignant - relatively balanced classes</li>
<li>This balance matters because accuracy alone can be misleading with imbalanced data</li>
</ul>
</div>

> **Key question**: With 569 samples and 30 features, which algorithm will perform best? Will a simple model (Logistic Regression) be enough, or do we need something more powerful (XGBoost)?

---

<a id="build-algorithms"></a>
## 3. Build 6 Competing Algorithms

We build 6 algorithms, each wrapped in a **scikit-learn Pipeline** with `StandardScaler` preprocessing. This ensures:
- Features are normalized (zero mean, unit variance) before training
- No data leakage - scaling is fitted only on training data during cross-validation

| Algorithm | Type | Key Hyperparameter |
|-----------|------|--------------------|
| Logistic Regression | Linear | max_iter=1000 |
| SVM (RBF) | Kernel-based | probability=True |
| KNN (k=5) | Distance-based | n_neighbors=5 |
| Decision Tree | Tree-based | max_depth=5 |
| Random Forest | Ensemble (bagging) | n_estimators=200 |
| XGBoost | Ensemble (boosting) | n_estimators=200, lr=0.1 |

<div class="alert alert-warning">
<strong>Why StandardScaler?</strong> Algorithms like SVM and KNN are distance-based - they are sensitive to feature scale. Without scaling, features with larger ranges (e.g., area) would dominate distance calculations over features with smaller ranges (e.g., smoothness).
</div>

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

algorithms = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=42))]),
    'SVM (RBF)': Pipeline([('scaler', StandardScaler()), ('model', SVC(kernel='rbf', probability=True, random_state=42))]),
    'KNN (k=5)': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=5))]),
    'Decision Tree': Pipeline([('scaler', StandardScaler()), ('model', DecisionTreeClassifier(max_depth=5, random_state=42))]),
    'Random Forest': Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier(n_estimators=200, max_features='sqrt', random_state=42, n_jobs=-1))]),
    'XGBoost': Pipeline([('scaler', StandardScaler()), ('model', XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42, eval_metric='logloss', verbosity=0))]),
}
print(f'Built {len(algorithms)} algorithm pipelines')

---

<a id="cross-val"></a>
## 4. Cross-Validation Comparison

Now we compare all 6 algorithms using **5-fold Stratified K-Fold** cross-validation. Stratified means each fold preserves the same class ratio (~63/37) as the full dataset.

We measure 5 metrics:

| Metric | What it measures |
|--------|-----------------|
| **Accuracy** | Overall correct predictions |
| **Precision** | Of all predicted benign, how many are truly benign? |
| **Recall** | Of all truly benign, how many did we catch? |
| **F1** | Harmonic mean of precision and recall |
| **AUC** | Area under ROC curve (rank-ordering quality) |

<div class="alert alert-info">
<strong>Watch for:</strong> Which algorithm wins each metric? Do different algorithms win different metrics?
</div>

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

print('=' * 75)
print(f'{"Algorithm":<22} {"Accuracy":>9} {"Precision":>10} {"Recall":>8} {"F1":>8} {"AUC":>8}')
print('=' * 75)

results = {}
for name, pipeline in algorithms.items():
    cv_results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    results[name] = {m: cv_results[f'test_{m}'].mean() for m in scoring}
    r = results[name]
    print(f'{name:<22} {r["accuracy"]:>8.3f}  {r["precision"]:>9.3f}  {r["recall"]:>7.3f}  {r["f1"]:>7.3f}  {r["roc_auc"]:>7.3f}')

print('=' * 75)

# Best per metric
print()
for metric in scoring:
    best = max(results, key=lambda a: results[a][metric])
    print(f'Best {metric}: {best} ({results[best][metric]:.4f})')

<div class="alert alert-success">
<strong>Key Observations:</strong>
<ul>
<li>All 6 algorithms score above 94% accuracy - this is a relatively clean dataset</li>
<li><strong>XGBoost</strong> typically leads on AUC (~0.995) and recall (~0.983)</li>
<li><strong>Decision Tree</strong> is consistently the weakest - it overfits even with max_depth=5</li>
<li>The performance gap between algorithms is small (~2-3%), which is common on clean tabular data</li>
</ul>
</div>

<div class="alert alert-warning">
<strong>Key insight:</strong> No single algorithm wins ALL metrics. XGBoost has the best AUC and recall, but the gap is narrow. For cancer screening, we prioritize <strong>recall</strong> (catching every cancer case), so XGBoost is our pick.
</div>

---

<a id="threshold-tuning"></a>
## 5. Threshold Tuning for Cancer Screening

By default, classifiers use a **0.5 probability threshold** - if `P(malignant) >= 0.5`, predict malignant. But for cancer screening, a missed malignant case (false negative) is far worse than a false alarm (false positive).

**Our goal**: Lower the malignant probability threshold to achieve **high malignant recall** - catching as many cancers as possible. The tradeoff is more false alarms (benign tumors flagged for biopsy), which is clinically acceptable.

<div class="alert alert-danger">
<strong>Think about it:</strong> A false negative means telling a cancer patient they're healthy. A false positive means one extra follow-up biopsy. Which mistake is more costly?
</div>

In [ ]:
from sklearn.metrics import precision_recall_curve, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

best_pipe = algorithms['XGBoost']
best_pipe.fit(X_train, y_train)

# For cancer screening: optimize MALIGNANT recall (catch maximum cancers)
# P(malignant) = column 0, P(benign) = column 1
y_proba_malign = best_pipe.predict_proba(X_test)[:, 0]

# ── Compare default vs tuned threshold ──
print('=== Default threshold (0.5) ===')
y_pred_default = np.where(y_proba_malign >= 0.5, 0, 1)  # 0=malignant, 1=benign
print(classification_report(y_test, y_pred_default, target_names=['malignant', 'benign']))

# ── Threshold tuning via precision-recall curve ──
# Compute PR curve for malignant class (flip labels so malignant=1)
precisions, recalls, thresholds = precision_recall_curve(
    (y_test == 0).astype(int),  # 1 = malignant (positive class)
    y_proba_malign               # P(malignant)
)

# With 42 malignant test samples, recall increments in ~2.4% steps (1/42)
# Find threshold for ~95% malignant recall (catches all but ~2 cancers)
target_recall = 0.95
idx = np.argmin(np.abs(recalls - target_recall))
opt_threshold = thresholds[idx] if idx < len(thresholds) else 0.5

print(f'\n=== Tuned threshold ({opt_threshold:.3f}) for ~{target_recall:.0%} malignant recall ===')
print(f'Interpretation: Flag as potentially malignant if P(malignant) >= {opt_threshold:.1%}')
print(f'Even a ~{opt_threshold:.0%} chance of cancer triggers a biopsy referral.\n')

y_pred_tuned = np.where(y_proba_malign >= opt_threshold, 0, 1)
print(classification_report(y_test, y_pred_tuned, target_names=['malignant', 'benign']))

# ── Threshold analysis at different recall targets ──
print('Threshold Analysis (malignant recall targets):')
print(f'{"Target":>8} {"Threshold":>10} {"Precision":>10} {"Actual Recall":>14}')
for target in [1.00, 0.95, 0.90, 0.85, 0.50]:
    i = np.argmin(np.abs(recalls - target))
    if i < len(thresholds):
        t = thresholds[i]
        p = precisions[i]
        r = recalls[i]
        marker = '  <-- tuned' if abs(target - target_recall) < 0.01 else ''
        print(f'{target:>7.0%} {t:>10.3f} {p:>10.3f} {r:>13.3f}{marker}')

<div class="alert alert-success">
<strong>How to read the threshold analysis:</strong>
<ul>
<li><strong>Default (0.5)</strong>: Optimizes for accuracy. Misses some malignant cases because borderline tumors are classified as benign.</li>
<li><strong>Tuned (~0.18)</strong>: Lowers the bar - flag as malignant if even ~18% chance of cancer. Catches ~95% of malignant cases.</li>
<li>The table shows the precision-recall tradeoff: lower threshold = higher recall (catch more cancers) but lower precision (more false alarms)</li>
</ul>
</div>

<div class="alert alert-info">
<strong>Real-world context:</strong> With only 42 malignant test samples, recall increments in ~2.4% steps (1/42). A production model would have thousands of samples for finer threshold tuning. The key principle remains: for screening, aggressively flag borderline cases for follow-up.
</div>

---

<a id="shap"></a>
## 6. SHAP Explainability

In healthcare, it is not enough to say "the model predicts malignant." Doctors need to know **why**.

<div class="alert alert-info">
<strong>What is SHAP?</strong><br>
<strong>SHAP (SHapley Additive exPlanations)</strong> uses game theory to assign each feature a contribution score for every prediction:
<ul>
<li><strong>Positive SHAP value</strong> = pushes prediction toward benign</li>
<li><strong>Negative SHAP value</strong> = pushes prediction toward malignant</li>
<li><strong>Larger absolute value</strong> = stronger influence</li>
</ul>
We use <code>TreeExplainer</code>, which is optimized for tree-based models like XGBoost and runs in polynomial time.
</div>

In [ ]:
import shap

# Train standalone XGBoost for SHAP
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42, eval_metric='logloss', verbosity=0)
model.fit(X_train_s, y_train)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_s)

# Global feature importance
mean_shap = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(mean_shap)[::-1][:10]

print('Global Top 10 Features (mean |SHAP value|):')
for rank, i in enumerate(top_idx, 1):
    print(f'  {rank:>2}. {feature_names[i]:<25} {mean_shap[i]:.4f}')

# Single prediction explanation
patient_idx = 0
patient_pred = model.predict_proba(X_test_s[patient_idx:patient_idx+1])[0]
true_label = 'benign' if y_test[patient_idx] else 'malignant'
print(f'\nPatient #{patient_idx}: P(malignant)={patient_pred[0]:.3f}, P(benign)={patient_pred[1]:.3f}')
print(f'True label: {true_label}')

patient_shap = shap_values[patient_idx]
top5 = np.argsort(np.abs(patient_shap))[::-1][:5]
print('\nTop 5 features driving this prediction:')
for i in top5:
    direction = '-> benign' if patient_shap[i] > 0 else '-> malignant'
    print(f'  {feature_names[i]:<25} SHAP={patient_shap[i]:+.4f} {direction}')

<div class="alert alert-success">
<strong>Key Observations:</strong>
<ul>
<li>The <strong>Global Top 10</strong> shows which features matter most across ALL patients</li>
<li><code>worst concave points</code> and <code>worst perimeter</code> dominate - these measure how irregular and large the worst-case cell nuclei are</li>
<li>All top features are clinically meaningful (morphological measurements), not data artifacts</li>
<li>The <strong>single patient explanation</strong> shows how each feature pushed that specific prediction toward benign or malignant</li>
</ul>
</div>

<div class="alert alert-warning">
<strong>Why this matters:</strong> If a model relied heavily on a feature like "patient ID" or "scan order," that would be a red flag for data leakage. Here, all top features are medically interpretable - a good sign for trustworthy ML.
</div>

---

<a id="shap-viz"></a>
## 7. SHAP Visualizations

Two complementary views of feature importance:

1. **Bar chart** - Shows the average absolute SHAP value per feature (overall importance)
2. **Beeswarm plot** - Shows how each feature's value (high=red, low=blue) affects the prediction. Each dot is one patient.

In [ ]:
# Summary plot (bar chart of feature importance)
shap.summary_plot(shap_values, X_test_s, feature_names=feature_names, plot_type='bar', show=True)

In [ ]:
# Beeswarm plot (shows direction of impact)
shap.summary_plot(shap_values, X_test_s, feature_names=feature_names, show=True)

<div class="alert alert-success">
<strong>How to read the beeswarm plot:</strong>
<ul>
<li>Each row is a feature, each dot is a patient</li>
<li><strong>Red dots</strong> = high feature value, <strong>Blue dots</strong> = low feature value</li>
<li>Dots on the <strong>right</strong> push the prediction toward benign, dots on the <strong>left</strong> push toward malignant</li>
<li>For <code>worst concave points</code>: red dots (high irregularity) cluster on the left (malignant), blue dots (low irregularity) on the right (benign) - this makes clinical sense</li>
</ul>
</div>

---

<a id="conclusion"></a>
## 8. Key Takeaways

<div class="alert alert-success">
<strong>1. No single algorithm wins all metrics.</strong><br>
The choice depends on your cost tradeoff:
<ul>
<li><strong>Cancer screening</strong> - Optimize malignant recall (miss no cancers) -> XGBoost with lowered threshold</li>
<li><strong>General classification</strong> - Optimize F1 or AUC -> XGBoost or SVM</li>
<li><strong>Regulatory review</strong> - Must be explainable -> SHAP reports for any model</li>
</ul>
</div>

<div class="alert alert-warning">
<strong>2. Threshold tuning is more impactful than algorithm selection.</strong><br>
Lowering the threshold from 0.5 to ~0.18 improved malignant recall from ~90% to ~95% - catching 2 additional cancers out of 42 in the test set. This single change matters more than which algorithm you pick.
</div>

<div class="alert alert-info">
<strong>3. Explainability builds trust.</strong><br>
SHAP tells you WHY the model made each prediction. All top features (<code>worst perimeter</code>, <code>worst concave points</code>, <code>worst area</code>) are clinically meaningful - the model learned real biology, not data artifacts.
</div>

---

### What to Try Next

- Add **Naive Bayes** as a 7th algorithm and see where it ranks
- Try **SMOTE** to simulate class imbalance (realistic for cancer screening)
- Build a **Streamlit dashboard** to let non-technical users interact with the model
- Add **confidence intervals** to the cross-validation results using bootstrap
- Deploy as a **REST API** with FastAPI for real-time predictions

---

<div class="alert alert-info">
<strong>Found this notebook useful?</strong> Give it an upvote on Kaggle! Have questions or suggestions? Leave a comment below or open an issue on <a href="https://github.com/genieincodebottle/aiml-companion">GitHub</a>.
</div>